# Stoa 02 — combine the datasets

Builds the analysis tables from the raw files that **`01_download.ipynb`** saved in `data/raw/`. Run that notebook first.

**The data model.** Each fact is stored once, in the table it belongs to:

| Table | One row per | Key | What's in it |
|---|---|---|---|
| `sensor_table.parquet` | sensor | `location_id` | location, `slope_pct`, `elevation_m` (from footpath steepness) |
| `hour_table.parquet` | UTC hour | `timestamp_utc` | `temp_c`, `rain_mm`, `wind_kmh`, `is_day` |
| `pcs_hourly.parquet` | sensor-hour | `location_id` + `timestamp_utc` | sensor latitude/longitude, pedestrian count (target), that hour's weather |

Slope and elevation never change, so they stay in `sensor_table` instead of being repeated on ~7 million hourly rows. Join them by `location_id` when you train (the last cell shows how). Latitude/longitude are the exception: they're copied onto every hourly row so each row says where it was counted.

**Steps** (each section defines its step, then runs it)

| Step | What it does | Writes |
|---|---|---|
| 1. Setup | folders, settings | — |
| 2. Counts | load the three count sources, fix Sep 2010, merge, convert to UTC | — |
| 3. Sensors | coordinates, nearest footpath for slope and elevation | `data/processed/sensor_table.parquet` |
| 4. Clean counts | drop impossible rows and sensors with no location, list gaps (needs step 3) | `data/interim/counts_clean.parquet`, `data/interim/missing_hours.parquet` |
| 5. Weather | one row per UTC hour | `data/processed/hour_table.parquet` |
| 6. Hourly table + checks | counts + lat/long + weather, then quality checks | `data/processed/pcs_hourly.parquet`, `data/processed/qa_report.json` |

Next: `03_preprocess.ipynb` fills the remaining gaps and makes the two final datasets. It does three things that are **not** done here: the LiDAR fallback for sensors with no footpath nearby, dropping indoor sensors, and flagging outage zeros. (The Sep 2010 fix stays here: the source merge needs the real hours, or it would delete those rows as duplicates.)

## 1. Setup

In [ ]:
%pip install -q shapely pyproj

In [ ]:
import os
import sys
from pathlib import Path

# Where the data lives. Colab: your Google Drive. A local clone of the repo: data/notebooks/,
# kept apart from the team pipeline's data/raw, data/interim and data/processed (git-ignored).
# Set STOA_NOTEBOOK_DATA to use another folder.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/stoa/data")
else:
    DATA = Path(os.environ.get("STOA_NOTEBOOK_DATA", Path.cwd().resolve().parent / "data" / "notebooks"))
DATA.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA)

In [ ]:
import calendar
import json
import zipfile

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import shapely
from pyproj import Transformer

### Settings

Everything you might want to change is here. The rest of the notebook just uses these names.

In [ ]:
# Optional date limits on the counts (Melbourne local dates, inclusive), e.g. "2009-05-01" / "2024-12-31".
# None keeps everything the raw files contain (May 2009 to the latest live pull).
DATE_FROM = None
DATE_TO = None

RAW = DATA / "raw"            # input: written by 01_download.ipynb, never edited
INTERIM = DATA / "interim"    # intermediate tables, useful for debugging
OUT = DATA / "processed"      # final tables

TZ = "Australia/Melbourne"
KEY = ["location_id", "sensing_date", "hourday"]   # how the raw sources identify one sensor-hour

# The three count sources overlap in places. When they disagree on the same sensor-hour,
# the lower number wins: live is the newest official copy, then the archive, then Wayback (the City's own
# portal export, archived by the Internet Archive; it matched the archive on ~98.5% of overlapping hours).
SOURCE_PRIORITY = {"live": 0, "archive": 1, "wayback": 2}
WAYBACK_COUNTS = "pcs_counts_wayback_20250502183548.csv"

# Footpath matching (all distances in metres).
MIN_SECTION_M = 5.0    # ignore footpath sections shorter than this: their grades are noise (0.5 m slivers report 500 %)
MAX_MATCH_M = 25.0     # a sensor further than this from any footpath gets empty slope/elevation here
                       # (03_preprocess fills them from the LiDAR elevation model)

# Weather: Open-Meteo name -> our column name.
# "Instant" values are measured at the hour mark. "Preceding" values are a total/average over
# the hour that ENDS at the timestamp; the counts are stamped at the START of their hour,
# so the preceding values are shifted back one hour to line up.
WEATHER_INSTANT = {
    "temperature_2m": "temp_c",
    "wind_speed_10m": "wind_kmh",
    "is_day": "is_day",
}
WEATHER_PRECEDING = {
    "precipitation": "rain_mm",
}

# Every count, fix and check is recorded here and saved as qa_report.json at the end.
qa: dict = {}

## 2. Counts

Three sources cover different years and overlap at the edges:

| Source | Covers | Format notes |
|---|---|---|
| archive | 2009-05 → 2022-10 | month as a name ("May"), `Sensor_ID`, no direction columns |
| Wayback | 2021-07 → 2025-05 | `;`-separated CSV, same columns as live |
| live | rolling ~2 years | parquet, one file per download date |

Each loader turns its source into the same columns: `location_id`, `sensing_date`, `hourday`, `pedestriancount`, `direction_1`, `direction_2`, `source`.

In [ ]:
def load_archive() -> pd.DataFrame:
    with zipfile.ZipFile(RAW / "pcs_archive_2009_2022.csv.zip") as z:
        # The zip also contains macOS "__MACOSX" metadata files; skip them.
        name = next(n for n in z.namelist() if n.endswith(".csv") and not n.startswith("__MACOSX"))
        with z.open(name) as f:
            cols = ["ID", "Year", "Month", "Mdate", "Time", "Sensor_ID", "Hourly_Counts"]
            a = pacsv.read_csv(f, convert_options=pacsv.ConvertOptions(include_columns=cols)).to_pandas()

    # The archive stores the month as a word ("January") and the day separately; rebuild a date.
    month = a["Month"].map({m: i for i, m in enumerate(calendar.month_name) if m}).astype("int64")
    a["sensing_date"] = pd.to_datetime(pd.DataFrame({"year": a.Year, "month": month, "day": a.Mdate}))

    # September 2010 bug: every row that month is stamped hour 0, so each sensor-day appears as
    # 24 copies of "00:00". The rows are still in hour order by ID, so numbering them 0..23 within
    # each sensor-day recovers the real hour. The assert stops the notebook if the pattern is different.
    dup = a.duplicated(["Sensor_ID", "sensing_date", "Time"], keep=False)
    bad = a.loc[dup].sort_values("ID")
    groups = bad.groupby(["Sensor_ID", "sensing_date"])
    assert bad["Time"].eq(0).all() and groups.size().eq(24).all(), "unexpected archive duplicates"
    a.loc[bad.index, "Time"] = groups.cumcount()
    qa["archive_rows"] = len(a)
    qa["archive_sep2010_rows_rehoured"] = len(bad)
    qa["archive_sep2010_extra_rows"] = int(len(bad) - groups.ngroups)

    return pd.DataFrame({
        "location_id": a.Sensor_ID.astype("int16"),   # same sensor IDs as location_id in the newer sources
        "sensing_date": a.sensing_date,
        "hourday": a.Time.astype("int8"),
        "pedestriancount": a.Hourly_Counts.astype("int32"),
        # The archive has no per-direction counts; fill with missing so all sources share columns.
        "direction_1": pd.Series(pd.NA, index=a.index, dtype="Int32"),
        "direction_2": pd.Series(pd.NA, index=a.index, dtype="Int32"),
        "source": "archive",
    })


def load_live() -> pd.DataFrame:
    # Every dated live download is kept. Newest file first, so when two downloads contain
    # the same sensor-hour, drop_duplicates(keep="first") keeps the newest value.
    frames = []
    for p in sorted(RAW.glob("pcs_live_counts_*.parquet"), reverse=True):
        cols = ["location_id", "sensing_date", "hourday", "direction_1", "direction_2", "pedestriancount"]
        frames.append(pd.read_parquet(p, columns=cols))
    live = pd.concat(frames, ignore_index=True).drop_duplicates(KEY, keep="first")
    qa["live_rows"] = len(live)
    qa["live_pulls"] = len(frames)
    return pd.DataFrame({
        "location_id": live.location_id.astype("int16"),
        "sensing_date": pd.to_datetime(live.sensing_date),
        "hourday": live.hourday.astype("int8"),
        "pedestriancount": live.pedestriancount.astype("int32"),
        "direction_1": live.direction_1.astype("Int32"),
        "direction_2": live.direction_2.astype("Int32"),
        "source": "live",
    })


def load_wayback() -> pd.DataFrame:
    # Portal CSV exports use ";" as the separator.
    cols = ["location_id", "sensing_date", "hourday", "direction_1", "direction_2", "pedestriancount"]
    w = pacsv.read_csv(
        RAW / WAYBACK_COUNTS,
        parse_options=pacsv.ParseOptions(delimiter=";"),
        convert_options=pacsv.ConvertOptions(include_columns=cols),
    ).to_pandas()
    qa["wayback_rows"] = len(w)
    return pd.DataFrame({
        "location_id": w.location_id.astype("int16"),
        "sensing_date": pd.to_datetime(w.sensing_date),
        "hourday": w.hourday.astype("int8"),
        "pedestriancount": w.pedestriancount.astype("int32"),
        "direction_1": w.direction_1.astype("Int32"),
        "direction_2": w.direction_2.astype("Int32"),
        "source": "wayback",
    })

### Merge the sources and convert to UTC

Counts are recorded in Melbourne clock time, which jumps at daylight saving. Weather is in UTC, which doesn't. So every count gets a `timestamp_utc` (used for joining) as well as `timestamp_local` (used for hour-of-day / weekday features).

- **October (clocks forward):** 02:00 doesn't exist. Those rows are mostly 0 fillers, so they're dropped.
- **April (clocks back):** 02:00 happens twice, but each sensor reports it once. It's treated as the first (daylight-saving) occurrence.

In [ ]:
def unify_counts() -> pd.DataFrame:
    c = pd.concat([load_live(), load_archive(), load_wayback()], ignore_index=True)

    # Optional date window from the settings.
    if DATE_FROM:
        c = c[c.sensing_date >= pd.Timestamp(DATE_FROM)]
    if DATE_TO:
        c = c[c.sensing_date <= pd.Timestamp(DATE_TO)]

    # Where sources overlap, sort so the preferred source comes first, then keep one row per sensor-hour.
    c["source"] = c["source"].astype("category")
    c["prio"] = c["source"].map(SOURCE_PRIORITY).astype("int8")
    c = c.sort_values(KEY + ["prio"]).drop_duplicates(KEY, keep="first").drop(columns="prio")
    qa["rows_after_source_merge"] = len(c)
    qa["rows_by_source"] = c["source"].value_counts().to_dict()

    # Local clock time -> timezone-aware -> UTC.
    # ambiguous=True picks the daylight-saving reading of the repeated April hour;
    # nonexistent="NaT" marks the skipped October hour so it can be dropped.
    local = pd.DatetimeIndex(c["sensing_date"] + pd.to_timedelta(c["hourday"], unit="h"))
    aware = local.tz_localize(TZ, ambiguous=np.ones(len(local), dtype=bool), nonexistent="NaT")
    keep = ~aware.isna()
    qa["dst_nonexistent_rows_dropped"] = int((~keep).sum())
    qa["dst_ambiguous_rows_assigned_daylight_time"] = int(
        (local.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT").isna() & keep).sum())
    c = c.loc[keep].copy()
    c["timestamp_local"] = aware[keep].astype(f"datetime64[ns, {TZ}]")
    c["timestamp_utc"] = c["timestamp_local"].dt.tz_convert("UTC")
    assert not c.duplicated(["location_id", "timestamp_utc"]).any(), "duplicate sensor-hours in UTC"
    return c.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

In [ ]:
# Run step 2: merged counts, one row per sensor-hour, with UTC timestamps.
INTERIM.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

counts = unify_counts()
print(f"counts: {len(counts):,} sensor-hours from {counts.location_id.nunique()} sensors")
print(counts.source.value_counts().to_string())

## 3. Sensors (the steepness table)

This is the only place latitude/longitude are used. Each sensor is matched **once** to its nearest footpath section, and the result is stored per sensor. The hourly table links to it later by `location_id`.

1. **Coordinates:** the current sensor table, plus the Oct 2022 snapshot for 18 retired sensors that the current table no longer lists.
2. **Nearest footpath:** coordinates are converted to metres (EPSG:7855, GDA2020 / MGA zone 55) so distances are real metres, then each sensor takes the nearest valid section's `gradepc` → `slope_pct` and the average of `rlmin`/`rlmax` → `elevation_m`.
   - Sections with a blank `gradepc` (4,455, never surveyed) and **slivers** shorter than 5 m (4,779) are skipped before matching; see *What the slope numbers mean* below.
3. **No footpath nearby:** a sensor more than 25 m from any footpath section (sensors 25 and 130 in the last run) would get a different street's slope, so its `slope_pct` and `elevation_m` are left **empty** and `slope_matched` is False. `03_preprocess.ipynb` fills them from the GA 5 m LiDAR elevation model.

### What the slope numbers mean

`slope_pct` (the footpath file's `gradepc`) is **how many metres you go up for every 100 metres you walk forward**:

```
                                          ● ← you end up 3 m higher
                              .  ·  ˙     ┃
                  .  ·  ˙                 ┃ 3 m up (rise)
      .  ·  ˙                             ┃
 ●━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
 start          100 m forward (run)

 slope % = rise ÷ run × 100 = 3 ÷ 100 × 100 = 3%
```

Only the ratio matters, so a 100 m section rising 3 m, a 50 m section rising 1.5 m and a 10 m section rising 0.3 m are all 3%. In the footpath file each section has its length (`distance`, the run) and its height at each end (`rlmin`, `rlmax`), so `gradepc = (rlmax − rlmin) ÷ distance × 100`. A typical section is 46 m long and rises 0.95 m: 0.95 ÷ 46 × 100 ≈ 2%.

| Slope | What it's like | Same slope as "1 in N" (`grade1in`) | In degrees |
|---|---|---|---|
| 0-2% | feels flat (most of the CBD; median sensor 1.7%) | 1 in 50+ | ≤ 1.1° |
| 5% | a gentle hill you'd notice on a bike | 1 in 20 | 2.9° |
| 8.3% | the steepest allowed for a wheelchair ramp | 1 in 12 | 4.8° |
| 20% | very steep: likely stairs or a ramp (sensor 139, 20.4%) | 1 in 5 | 11.3° |
| 100% | 45°: 1 m up for every 1 m forward | 1 in 1 | 45° |

Percent is kept because it's the city's own unit and easy to sanity-check; at street slopes percent and degrees are almost proportional, so a model learns the same from either. "1 in N" runs backwards (bigger N = flatter) and goes to infinity on flat ground, so it isn't used.

**Why slivers are skipped:** grade divides by the length, so a tiny length makes a huge grade. A 1.32 m piece rising 7.64 m gives 7.64 ÷ 1.32 × 100 = **580%**, steeper than a wall: a data artefact where sections join, not a real slope. Sections shorter than 5 m (`MIN_SECTION_M`) are ignored for that reason.

In [ ]:
def sensor_coordinates(location_ids) -> pd.DataFrame:
    cur = pd.read_parquet(RAW / "pcs_sensor_locations.parquet").drop(columns="location")
    cur["coord_source"] = "sensor_locations_current"

    # Retired sensors are gone from the current table; take them from the Oct 2022 copy.
    # That copy calls the ID "sensor_id"; it's the same number as location_id.
    old = pd.DataFrame(json.loads((RAW / "pcs_sensor_locations_2022-10-24_wayback.json").read_text("utf-8")))
    old = pd.DataFrame({
        "location_id": old.sensor_id.astype(int),
        "sensor_description": old.sensor_description,
        "sensor_name": old.sensor_name,
        "installation_date": pd.to_datetime(old.installation_date).dt.date,
        "status": old.status,
        "latitude": old.latitude.astype(float),
        "longitude": old.longitude.astype(float),
        "coord_source": "sensor_locations_2022_snapshot",
    })
    s = pd.concat([cur, old[~old.location_id.isin(cur.location_id)]], ignore_index=True)

    # Last resort: the newer count exports carry a "lat, lon" string on every row.
    missing = sorted(set(location_ids) - set(s.location_id))
    if missing:
        w = pacsv.read_csv(RAW / WAYBACK_COUNTS, parse_options=pacsv.ParseOptions(delimiter=";"),
                           convert_options=pacsv.ConvertOptions(include_columns=["location_id", "sensor_name", "location"])
                           ).to_pandas().dropna().drop_duplicates("location_id")
        w = w[w.location_id.isin(missing)]
    if missing and len(w):
        latlon = w.location.str.split(",", expand=True).astype(float)
        s = pd.concat([s, pd.DataFrame({
            "location_id": w.location_id, "sensor_name": w.sensor_name,
            "latitude": latlon[0], "longitude": latlon[1], "coord_source": "count_export_rows",
        })], ignore_index=True)

    # Only keep sensors that actually appear in the counts.
    s = s[s.location_id.isin(location_ids)].copy()
    s["location_id"] = s.location_id.astype("int16")
    return s

In [ ]:
def match_footpaths(s: pd.DataFrame) -> pd.DataFrame:
    # Lat/long (degrees) -> MGA zone 55 (metres), so "25 m" really means 25 m.
    to_mga = Transformer.from_crs("EPSG:4326", "EPSG:7855", always_xy=True)
    fc = json.loads((RAW / "footpath_steepness.geojson").read_text("utf-8"))["features"]
    props = pd.DataFrame([f["properties"] for f in fc])
    geoms = np.array([shapely.geometry.shape(f["geometry"]) if f["geometry"] else None for f in fc], dtype=object)

    # Usable sections: have a shape, a grade, both heights, and are at least MIN_SECTION_M long.
    # Filtering BEFORE matching means a sensor never lands on a section with a blank grade.
    valid = (pd.notna(geoms) & props.gradepc.notna() & props.rlmin.notna() & props.rlmax.notna()
             & (props.distance >= MIN_SECTION_M)).to_numpy()
    props, geoms = props[valid].reset_index(drop=True), geoms[valid]
    geoms = shapely.transform(geoms, lambda xy: np.column_stack(to_mga.transform(xy[:, 0], xy[:, 1])))
    qa["footpath_sections_total"] = len(fc)
    qa["footpath_sections_used"] = int(valid.sum())

    # STRtree is a spatial index: it finds the nearest polygon for every sensor point quickly.
    # pi = which sensor, gi = which footpath section, dist = metres between them (0 = sensor is on it).
    tree = shapely.STRtree(geoms)
    x, y = to_mga.transform(s.longitude.to_numpy(), s.latitude.to_numpy())
    pts = shapely.points(x, y)
    (pi, gi), dist = tree.query_nearest(pts, return_distance=True, all_matches=False)
    near = props.iloc[gi].reset_index(drop=True)
    order = np.argsort(pi)   # put the matches back in the same order as the sensors

    out = s.copy()
    out["easting_m"], out["northing_m"] = x, y
    out["slope_pct"] = near.gradepc.to_numpy()[order]
    out["elevation_m"] = ((near.rlmin + near.rlmax) / 2).to_numpy()[order]   # midpoint of the section's lowest and highest point
    out["match_distance_m"] = dist[order]
    out["footpath_mcc_id"] = near.mcc_id.to_numpy()[order]   # which footpath section was used (links back to the raw file)
    out["footpath_address"] = near.address.to_numpy()[order]
    return out

In [ ]:
def drop_far_matches(s: pd.DataFrame) -> pd.DataFrame:
    # A sensor too far from any footpath was matched to some other street: blank its values
    # (03_preprocess fills them from LiDAR). match_distance_m and the section id are kept for reference.
    far = s.match_distance_m > MAX_MATCH_M
    s["slope_matched"] = ~far   # True = slope/elevation came from a footpath section within MAX_MATCH_M
    s.loc[far, ["slope_pct", "elevation_m"]] = np.nan
    s["place_source"] = np.where(far, None, "footpath_steepness")
    return s


def build_sensor_table(location_ids) -> pd.DataFrame:
    s = sensor_coordinates(location_ids)
    s = drop_far_matches(match_footpaths(s.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)))
    qa["sensors"] = len(s)
    qa["sensors_by_coord_source"] = s.coord_source.value_counts().to_dict()
    qa["sensors_without_footpath"] = s.loc[~s.slope_matched, "location_id"].tolist()
    qa["sensors_slope_over_10pct"] = s.loc[s.slope_pct > 10, ["location_id", "slope_pct"]].values.tolist()
    qa["slope_pct_summary"] = s.slope_pct.describe().round(2).to_dict()
    cols = ["location_id", "sensor_description", "sensor_name", "location_type", "status", "installation_date",
            "latitude", "longitude", "easting_m", "northing_m", "coord_source",
            "slope_pct", "elevation_m", "match_distance_m", "slope_matched", "place_source",
            "footpath_mcc_id", "footpath_address"]
    return s[cols].sort_values("location_id").reset_index(drop=True)

In [ ]:
# Run step 3: the steepness table (one row per sensor). Built from the sensors that appear in the counts.
sensors = build_sensor_table(counts.location_id.unique())
sensors.to_parquet(OUT / "sensor_table.parquet", index=False)
print(f"sensor table: {len(sensors)} sensors; no footpath within {MAX_MATCH_M:.0f} m (left empty): "
      f"{qa['sensors_without_footpath']}")

## 4. Clean the counts

This needs the sensor table from step 3, so it comes after it.


- Drop impossible values (negative counts) and counts from sensors with no known location (they can't be placed on a footpath).
- Check that `direction_1 + direction_2 = pedestriancount` wherever directions exist.
- **Missing hours** (no row at all) are never filled with 0; `missing_hours` lists every gap instead.

Indoor sensors and outage zeros are handled in `03_preprocess.ipynb`.

In [ ]:
def clean_counts(c: pd.DataFrame, sensors: pd.DataFrame) -> pd.DataFrame:
    qa["negative_counts"] = int((c.pedestriancount < 0).sum())
    c = c[c.pedestriancount >= 0]

    # The two directions should add up to the total. Only newer sources have directions.
    has_dir = c.direction_1.notna() & c.direction_2.notna()
    qa["direction_rows_checked"] = int(has_dir.sum())
    qa["direction_sum_mismatches"] = int(
        (c.direction_1[has_dir] + c.direction_2[has_dir] != c.pedestriancount[has_dir]).sum())

    # Keep only sensors that made it into the sensor table (i.e. have coordinates).
    no_coords = sensors.loc[sensors.latitude.isna(), "location_id"]
    unknown = ~c.location_id.isin(sensors.location_id)
    drop = c.location_id.isin(no_coords) | unknown
    qa["rows_dropped_no_coordinates"] = int(drop.sum())
    c = c[~drop].copy()
    qa["rows_clean"] = len(c)
    return c.reset_index(drop=True)


def missing_hours(c: pd.DataFrame) -> pd.DataFrame:
    # Compare each row with the previous row of the same sensor; more than 1 hour apart = a gap.
    prev = c.groupby("location_id").timestamp_utc.shift()
    gap = (c.timestamp_utc - prev) > pd.Timedelta(hours=1)
    g = pd.DataFrame({
        "location_id": c.location_id[gap],
        "gap_start_utc": prev[gap] + pd.Timedelta(hours=1),   # first missing hour
        "gap_end_utc": c.timestamp_utc[gap] - pd.Timedelta(hours=1),   # last missing hour
    })
    g["n_hours"] = ((g.gap_end_utc - g.gap_start_utc) / pd.Timedelta(hours=1) + 1).astype("int64")
    qa["missing_hour_gaps"] = len(g)
    qa["missing_hours_total"] = int(g.n_hours.sum())
    return g.reset_index(drop=True)

In [ ]:
# Run step 4: clean the counts (uses the sensor table from step 3) and list the gaps.
counts = clean_counts(counts, sensors)
counts.to_parquet(INTERIM / "counts_clean.parquet", index=False)
missing_hours(counts).to_parquet(INTERIM / "missing_hours.parquet", index=False)
print(f"clean counts: {len(counts):,} | missing hours: {qa['missing_hours_total']:,}")

## 5. Weather

One row per UTC hour for central Melbourne. The study area is only a few kilometres across, so every sensor shares the same weather for a given hour.

In [ ]:
def build_hour_table() -> pd.DataFrame:
    # One JSON per year; stack them. drop_duplicates guards against a year boundary being saved twice.
    frames = [pd.DataFrame(json.loads(p.read_text())["hourly"])
              for p in sorted((RAW / "open_meteo_era5").glob("era5_hourly_*.json"))]
    w = pd.concat(frames, ignore_index=True).drop_duplicates("time")
    w.index = pd.to_datetime(w.pop("time")).dt.tz_localize("UTC").rename("timestamp_utc")
    w = w.sort_index()
    assert w.index.to_series().diff().dropna().eq(pd.Timedelta(hours=1)).all(), "weather hours not contiguous"

    # Line up "preceding hour" values with the count hour (see Settings): the value stamped
    # 10:00 describes 09:00-10:00, so it moves to the 09:00 row.
    w[list(WEATHER_PRECEDING)] = w[list(WEATHER_PRECEDING)].shift(-1)
    w = w[[*WEATHER_INSTANT, *WEATHER_PRECEDING]].rename(columns={**WEATHER_INSTANT, **WEATHER_PRECEDING})
    w = w.astype("float32").astype({"is_day": "Int8"})
    w.index = w.index.astype("datetime64[ns, UTC]")   # same time precision as the counts, so the join matches

    # ERA5 runs ~6 days behind, so the newest hours are empty. Count hours after this get no weather.
    qa["weather_hours"] = len(w)
    qa["weather_last_non_null_utc"] = str(w.temp_c.last_valid_index())
    return w.reset_index()

In [ ]:
# Run step 5: the weather table (one row per UTC hour).
hours = build_hour_table()
hours.to_parquet(OUT / "hour_table.parquet", index=False)
print(f"weather: {len(hours):,} hours, last with data {qa['weather_last_non_null_utc']}")

## 6. Hourly table and checks

`pcs_hourly.parquet` = sensor latitude/longitude + counts + weather, one row per sensor-hour. **No slope or elevation here**; those stay in `sensor_table`.

It's written one year at a time so Colab doesn't run out of memory.

In [ ]:
def write_hourly(c: pd.DataFrame, sensors: pd.DataFrame, hours: pd.DataFrame) -> int:
    # Direction columns are left out: they add up to the target, so a model could "cheat" with them.
    base = c[["location_id", "timestamp_utc", "timestamp_local", "sensing_date", "hourday",
              "pedestriancount", "source"]]
    place = sensors[["location_id", "latitude", "longitude"]]   # where each sensor is
    out = OUT / "pcs_hourly.parquet"
    writer, schema, n = None, None, 0
    for _, part in base.groupby(base.timestamp_utc.dt.year, sort=True):
        # Left joins: every count row is kept. Lat/long come from the sensor table (by location_id),
        # weather from the hour table (by timestamp_utc; empty if that hour has no weather).
        j = part.merge(place, on="location_id", how="left", validate="many_to_one")
        j = j.merge(hours, on="timestamp_utc", how="left", validate="many_to_one")
        assert len(j) == len(part), "join changed the row count"
        j = j.sort_values(["timestamp_utc", "location_id"])
        j["sensing_date"] = j.sensing_date.dt.date
        j["source"] = j.source.astype(str)
        t = pa.Table.from_pandas(j, preserve_index=False)
        if writer is None:   # the first year fixes the column types; later years are cast to match
            schema = t.schema
            writer = pq.ParquetWriter(out, schema, compression="zstd")
        writer.write_table(t.cast(schema))
        n += len(j)
    writer.close()
    return n


def validate(path: Path, n_clean: int, sensors: pd.DataFrame) -> None:
    t = pq.read_table(path, columns=["location_id", "latitude", "longitude", "timestamp_utc", "hourday",
                                      "sensing_date", "pedestriancount", "temp_c", "is_day"]).to_pandas()
    checks = {
        "hourly_rows_equal_clean_rows": len(t) == n_clean,
        "no_duplicate_sensor_hours": not t.duplicated(["location_id", "timestamp_utc"]).any(),
        "no_negative_counts": bool((t.pedestriancount >= 0).all()),
        "direction_sum_matches_total": qa["direction_sum_mismatches"] == 0,
        # The two tables must link cleanly: one row per sensor, and every hourly sensor has one.
        "sensor_table_one_row_per_sensor": not sensors.location_id.duplicated().any(),
        "every_hourly_sensor_in_sensor_table": bool(t.location_id.isin(sensors.location_id).all()),
        "every_hourly_row_has_lat_long": bool(t.latitude.notna().all() and t.longitude.notna().all()),
        # Slope/elevation are empty exactly for the sensors with no footpath nearby (filled in 03_preprocess).
        "slope_empty_only_where_no_footpath": bool(
            (sensors.slope_pct.isna() == ~sensors.slope_matched).all()
            and (sensors.elevation_m.isna() == ~sensors.slope_matched).all()),
    }

    # Sanity checks that the weather lines up with the right hours. If UTC conversion or the join
    # were off by several hours, these would fail.
    month = pd.to_datetime(t.sensing_date).dt.month
    summer = month.isin([12, 1, 2])
    aft = t.temp_c[summer & t.hourday.between(14, 16)].mean()
    night = t.temp_c[summer & t.hourday.between(2, 4)].mean()
    checks["summer_afternoon_warmer_than_night"] = bool(aft > night + 3)
    qa["summer_afternoon_vs_night_temp_c"] = [round(float(aft), 2), round(float(night), 2)]
    checks["daylight_at_local_noon"] = bool(t.is_day[t.hourday == 12].dropna().eq(1).mean() > 0.999)
    checks["dark_at_local_midnight"] = bool(t.is_day[t.hourday == 0].dropna().eq(0).mean() > 0.999)

    qa["rows_without_weather"] = int(t.temp_c.isna().sum())   # expected: only the last few days (ERA5 lag)
    qa["checks"] = checks
    qa["final_rows"] = len(t)
    qa["final_sensors"] = int(t.location_id.nunique())
    qa["final_range_utc"] = [str(t.timestamp_utc.min()), str(t.timestamp_utc.max())]

In [ ]:
# Run step 6: write the hourly table, then check it and save the QA report.
n = write_hourly(counts, sensors, hours)
print(f"hourly table: {n:,} rows")

validate(OUT / "pcs_hourly.parquet", len(counts), sensors)
(OUT / "qa_report.json").write_text(json.dumps(qa, indent=2, default=str))
print(json.dumps(qa, indent=2, default=str))

failed = [k for k, ok in qa["checks"].items() if not ok]
if failed:
    raise RuntimeError(f"checks failed: {failed}")

## 7. Look at the results

In [ ]:
# The steepness table: one row per sensor.
sensors[["location_id", "sensor_name", "slope_pct", "elevation_m", "match_distance_m", "slope_matched", "place_source"]]

In [ ]:
# The first rows of the hourly table (reads one chunk, not all ~7 million rows).
f = pq.ParquetFile(OUT / "pcs_hourly.parquet")
print(f.metadata.num_rows, "rows,", f.metadata.num_columns, "columns")
f.read_row_group(0).to_pandas().head()

## 8. Joining for training

For training, use the two final datasets from `03_preprocess.ipynb`; this shows the same join on combine's tables. Add slope and elevation only when you need them, by `location_id`. `validate="many_to_one"` makes pandas stop if a sensor somehow appears twice in the sensor table.

In [ ]:
# Example on one year to keep memory low; drop the filter to use everything.
hourly = pd.read_parquet(OUT / "pcs_hourly.parquet", filters=[("timestamp_utc", ">=", pd.Timestamp("2024-01-01", tz="UTC"))])
place = pd.read_parquet(OUT / "sensor_table.parquet", columns=["location_id", "slope_pct", "elevation_m"])

train = hourly.merge(place, on="location_id", how="left", validate="many_to_one")
assert len(train) == len(hourly)
train.head()